In [4]:
import math

from incidentiq.search import SearchEngine

In [5]:
engine = SearchEngine(
    "../data/processed/logs.parquet"
)

Batches: 100%|██████████| 63/63 [00:05<00:00, 11.97it/s]


In [6]:
RELEVANCE_IRRELEVANT = 0
RELEVANCE_SOMEWHAT = 1
RELEVANCE_STRONG = 2

In [9]:
judgments = qrels["hardware stopped working"]

print("Total judgments:", len(judgments))

print(
    "Strong:",
    sum(score == 2 for score in judgments.values())
)

print(
    "Somewhat:",
    sum(score == 1 for score in judgments.values())
)

print(
    "Irrelevant:",
    sum(score == 0 for score in judgments.values())
)

Total judgments: 50
Strong: 18
Somewhat: 30
Irrelevant: 2


In [10]:
for doc_id, relevance in judgments.items():

    print(
        f"doc={doc_id:4} | relevance={relevance}"
    )

doc=1329 | relevance=2
doc=1216 | relevance=2
doc=1219 | relevance=2
doc=1230 | relevance=2
doc=1407 | relevance=2
doc=1948 | relevance=2
doc=1218 | relevance=2
doc= 457 | relevance=2
doc=1227 | relevance=2
doc=1229 | relevance=2
doc= 620 | relevance=2
doc= 450 | relevance=2
doc= 231 | relevance=2
doc= 232 | relevance=2
doc= 233 | relevance=2
doc= 234 | relevance=2
doc= 238 | relevance=2
doc=1029 | relevance=2
doc=1406 | relevance=1
doc= 521 | relevance=1
doc=1221 | relevance=1
doc=1224 | relevance=1
doc=1220 | relevance=1
doc=1739 | relevance=1
doc=1747 | relevance=1
doc=1756 | relevance=1
doc=1259 | relevance=1
doc=1519 | relevance=1
doc=1258 | relevance=1
doc=1202 | relevance=1
doc=1985 | relevance=1
doc=1982 | relevance=1
doc=1962 | relevance=1
doc=1745 | relevance=1
doc=1480 | relevance=1
doc=1459 | relevance=1
doc=1483 | relevance=1
doc=1373 | relevance=1
doc=1495 | relevance=1
doc=1281 | relevance=1
doc=1069 | relevance=1
doc=1456 | relevance=1
doc= 314 | relevance=1
doc= 837 | 

In [11]:
def get_relevance_list(results, qrels):

    return [
        qrels.get(
            result["doc_id"],
            0
        )
        for result in results
    ]

In [12]:
qrels = {
    "hardware stopped working": {
        1329: 2,
        1216: 2,
        1219: 2,
        1230: 2,
        1407: 2,

        1948: 2,
        1218: 2,
        457: 2,
        1227: 2,
        1229: 2,
        620: 2,

        450: 2,

        231: 2,
        232: 2,
        233: 2,
        234: 2,
        238: 2,

        1029: 2,

        1406: 1,
        521: 1,
        1221: 1,
        1224: 1,
        1220: 1,

        1739: 1,
        1747: 1,
        1756: 1,
        1259: 1,
        1519: 1,

        1258: 1,
        1202: 1,

        1985: 1,
        1982: 1,
        1962: 1,
        1745: 1,
        1480: 1,
        1459: 1,
        1483: 1,
        1373: 1,
        1495: 1,

        1281: 1,
        1069: 1,
        1456: 1,
        314: 1,
        837: 1,
        1234: 1,
        1514: 1,
        1261: 1,
        1402: 1,

        1370: 0,
        1371: 0,
    },

    "network connection failure": {
        1783: 2,
        1777: 2,
        1780: 2,
        1778: 2,
        1782: 2,
        1784: 2,

        1749: 2,
        1757: 2,
        1748: 2,
        1520: 2,
        1718: 2,

        1231: 2,

        295: 0,

        1216: 0,
        1219: 0,
        1230: 0,
        1407: 0,
        1329: 0,

        1774: 0,
        1773: 0,
    }
}

In [13]:
def retrieve_all(query, top_k=10):
    return {
        "BM25": engine.search_bm25(
            query,
            top_k=top_k
        ),

        "Semantic": engine.search_semantic(
            query,
            top_k=top_k
        ),

        "RRF": engine.search_hybrid(
            query,
            top_k=top_k
        ),
    }

In [14]:
def get_relevance_list(results, qrels):
    return [
        qrels.get(
            result["doc_id"],
            0
        )
        for result in results
    ]

In [15]:
def precision_at_k(results, qrels, k):
    top_results = results[:k]

    relevant = sum(
        qrels.get(
            result["doc_id"],
            0
        ) > 0
        for result in top_results
    )

    return relevant / k

In [16]:
def dcg_at_k(relevance_scores, k):
    relevance_scores = relevance_scores[:k]

    dcg = 0.0

    for rank, relevance in enumerate(
        relevance_scores,
        start=1
    ):
        dcg += (
            (2 ** relevance - 1)
            / math.log2(rank + 1)
        )

    return dcg

In [17]:
def ndcg_at_k(relevance_scores, k):
    actual_scores = relevance_scores[:k]

    ideal_scores = sorted(
        relevance_scores,
        reverse=True
    )[:k]

    actual_dcg = dcg_at_k(
        actual_scores,
        k
    )

    ideal_dcg = dcg_at_k(
        ideal_scores,
        k
    )

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg

In [18]:
def evaluate_query(query, top_k=10):
    results = retrieve_all(
        query,
        top_k=top_k
    )

    query_qrels = qrels[query]

    evaluation = {}

    for system_name, search_results in results.items():
        relevance = get_relevance_list(
            search_results,
            query_qrels
        )

        evaluation[system_name] = {
            "precision": precision_at_k(
                search_results,
                query_qrels,
                top_k
            ),

            "ndcg": ndcg_at_k(
                relevance,
                top_k
            ),

            "relevance": relevance,

            "results": search_results,
        }

    return evaluation

In [19]:
evaluated_queries = {}

for query in qrels:
    evaluated_queries[query] = evaluate_query(
        query,
        top_k=10
    )

In [20]:
for query, systems in evaluated_queries.items():

    print()
    print("=" * 60)
    print(f"QUERY: {query}")
    print("=" * 60)

    for system_name, metrics in systems.items():

        print(
            f"{system_name:10} | "
            f"P@10={metrics['precision']:.2f} | "
            f"nDCG@10={metrics['ndcg']:.4f}"
        )


QUERY: hardware stopped working
BM25       | P@10=0.00 | nDCG@10=0.0000
Semantic   | P@10=1.00 | nDCG@10=1.0000
RRF        | P@10=1.00 | nDCG@10=1.0000

QUERY: network connection failure
BM25       | P@10=1.00 | nDCG@10=1.0000
Semantic   | P@10=1.00 | nDCG@10=1.0000
RRF        | P@10=1.00 | nDCG@10=1.0000


In [21]:
for query, systems in evaluated_queries.items():

    print()
    print(f"QUERY: {query}")

    for system_name, metrics in systems.items():

        print(
            f"{system_name:10} | "
            f"{metrics['relevance']}"
        )


QUERY: hardware stopped working
BM25       | []
Semantic   | [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
RRF        | [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

QUERY: network connection failure
BM25       | [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
Semantic   | [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
RRF        | [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


In [22]:
def show_results(results):
    for rank, result in enumerate(
        results,
        start=1
    ):
        score = result.get(
            "score",
            result.get(
                "semantic_score",
                0.0
            )
        )

        print(
            f"{rank:2} | "
            f"doc={result['doc_id']} | "
            f"score={score:.4f} | "
            f"{result['message']}"
        )

In [23]:
query = "hardware stopped working"

results = retrieve_all(
    query,
    top_k=10
)

for system_name, search_results in results.items():

    print()
    print("=" * 20, system_name, "=" * 20)

    show_results(search_results)


==================== BM25 ====================

==================== Semantic ====================
 1 | doc=1329 | score=0.3561 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is asserted. Temperature Mask is not active. No temperature error. Temperature Limit Error Latch is clear. PGOOD IS NOT ASSERTED. PGOOD ERROR LATCH IS ACTIVE. MPGOOD IS NOT OK. MPGOOD ERROR LATCH IS ACTIVE. The 2.5 volt rail is OK. The 1.5 volt rail is OK.
 2 | doc=1219 | score=0.3561 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is asserted. Temperature Mask is not active. No temperature error. Temperature Limit Error Latch is clear. PGOOD IS NOT ASSERTED. PGOOD ERROR LATCH IS ACTIVE. MPGOOD IS NOT OK. MPGOOD ERROR LATCH IS ACTIVE. The 2.5 volt rail is OK. The 1.5 volt rail is OK.
 3 | doc=1407 | score=0.3561 | Node card status: no ALERTs are act

In [24]:
remaining_queries = [
    "cache parity error",
    "machine malfunction",
]

candidate_results = {}

for query in remaining_queries:

    candidate_results[query] = retrieve_all(
        query,
        top_k=50
    )

In [25]:
for query in remaining_queries:

    print()
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    print("\nBM25")
    print("-" * 70)

    show_results(
        candidate_results[query]["BM25"]
    )


QUERY: cache parity error

BM25
----------------------------------------------------------------------
 1 | doc=0 | score=10.5117 | instruction cache parity error corrected
 2 | doc=1 | score=10.5117 | instruction cache parity error corrected
 3 | doc=2 | score=10.5117 | instruction cache parity error corrected
 4 | doc=3 | score=10.5117 | instruction cache parity error corrected
 5 | doc=58 | score=10.5117 | instruction cache parity error corrected
 6 | doc=59 | score=10.5117 | instruction cache parity error corrected
 7 | doc=60 | score=10.5117 | instruction cache parity error corrected
 8 | doc=61 | score=10.5117 | instruction cache parity error corrected
 9 | doc=62 | score=10.5117 | instruction cache parity error corrected
10 | doc=63 | score=10.5117 | instruction cache parity error corrected
11 | doc=64 | score=10.5117 | instruction cache parity error corrected
12 | doc=65 | score=10.5117 | instruction cache parity error corrected
13 | doc=316 | score=10.5117 | instruction cache

In [26]:
for query in remaining_queries:

    print()
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    print("\nSEMANTIC")
    print("-" * 70)

    show_results(
        candidate_results[query]["Semantic"]
    )


QUERY: cache parity error

SEMANTIC
----------------------------------------------------------------------
 1 | doc=0 | score=0.8309 | instruction cache parity error corrected
 2 | doc=1 | score=0.8309 | instruction cache parity error corrected
 3 | doc=2 | score=0.8309 | instruction cache parity error corrected
 4 | doc=3 | score=0.8309 | instruction cache parity error corrected
 5 | doc=1997 | score=0.8309 | instruction cache parity error corrected
 6 | doc=320 | score=0.8309 | instruction cache parity error corrected
 7 | doc=60 | score=0.8309 | instruction cache parity error corrected
 8 | doc=63 | score=0.8309 | instruction cache parity error corrected
 9 | doc=62 | score=0.8309 | instruction cache parity error corrected
10 | doc=59 | score=0.8309 | instruction cache parity error corrected
11 | doc=1996 | score=0.8309 | instruction cache parity error corrected
12 | doc=61 | score=0.8309 | instruction cache parity error corrected
13 | doc=1998 | score=0.8309 | instruction cache pa

In [27]:
for query in remaining_queries:

    print()
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    print("\nRRF")
    print("-" * 70)

    show_results(
        candidate_results[query]["RRF"]
    )


QUERY: cache parity error

RRF
----------------------------------------------------------------------
 1 | doc=0 | score=0.0328 | instruction cache parity error corrected
 2 | doc=1 | score=0.0323 | instruction cache parity error corrected
 3 | doc=2 | score=0.0317 | instruction cache parity error corrected
 4 | doc=3 | score=0.0312 | instruction cache parity error corrected
 5 | doc=60 | score=0.0299 | instruction cache parity error corrected
 6 | doc=59 | score=0.0294 | instruction cache parity error corrected
 7 | doc=63 | score=0.0290 | instruction cache parity error corrected
 8 | doc=62 | score=0.0290 | instruction cache parity error corrected
 9 | doc=61 | score=0.0286 | instruction cache parity error corrected
10 | doc=320 | score=0.0281 | instruction cache parity error corrected
11 | doc=58 | score=0.0279 | instruction cache parity error corrected
12 | doc=317 | score=0.0268 | instruction cache parity error corrected
13 | doc=319 | score=0.0261 | instruction cache parity erro

In [28]:
candidate_pools = {}

for query in remaining_queries:

    documents = {}

    for system_results in candidate_results[query].values():

        for result in system_results:

            doc_id = result["doc_id"]

            documents[doc_id] = result

    candidate_pools[query] = list(
        documents.values()
    )

    print(
        f"{query}: "
        f"{len(candidate_pools[query])} candidates"
    )

cache parity error: 51 candidates
machine malfunction: 55 candidates


In [29]:
for query, candidates in candidate_pools.items():

    print()
    print("=" * 80)
    print(f"QUERY: {query}")
    print(f"CANDIDATES: {len(candidates)}")
    print("=" * 80)

    for rank, result in enumerate(candidates, start=1):

        print(
            f"{rank:2} | "
            f"doc={result['doc_id']} | "
            f"{result['message']}"
        )


QUERY: cache parity error
CANDIDATES: 51
 1 | doc=0 | instruction cache parity error corrected
 2 | doc=1 | instruction cache parity error corrected
 3 | doc=2 | instruction cache parity error corrected
 4 | doc=3 | instruction cache parity error corrected
 5 | doc=58 | instruction cache parity error corrected
 6 | doc=59 | instruction cache parity error corrected
 7 | doc=60 | instruction cache parity error corrected
 8 | doc=61 | instruction cache parity error corrected
 9 | doc=62 | instruction cache parity error corrected
10 | doc=63 | instruction cache parity error corrected
11 | doc=64 | instruction cache parity error corrected
12 | doc=65 | instruction cache parity error corrected
13 | doc=316 | instruction cache parity error corrected
14 | doc=317 | instruction cache parity error corrected
15 | doc=318 | instruction cache parity error corrected
16 | doc=319 | instruction cache parity error corrected
17 | doc=320 | instruction cache parity error corrected
18 | doc=321 | instruc

In [30]:
candidates = candidate_pools["cache parity error"]

for rank, result in enumerate(candidates, start=1):
    if 21 <= rank <= 51:
        print(
            f"{rank:2} | "
            f"doc={result['doc_id']} | "
            f"{result['message']}"
        )

21 | doc=327 | instruction cache parity error corrected
22 | doc=328 | instruction cache parity error corrected
23 | doc=329 | instruction cache parity error corrected
24 | doc=330 | instruction cache parity error corrected
25 | doc=331 | instruction cache parity error corrected
26 | doc=333 | instruction cache parity error corrected
27 | doc=334 | instruction cache parity error corrected
28 | doc=346 | instruction cache parity error corrected
29 | doc=370 | instruction cache parity error corrected
30 | doc=372 | instruction cache parity error corrected
31 | doc=430 | instruction cache parity error corrected
32 | doc=1526 | instruction cache parity error corrected
33 | doc=1689 | instruction cache parity error corrected
34 | doc=1692 | instruction cache parity error corrected
35 | doc=1991 | instruction cache parity error corrected
36 | doc=1992 | instruction cache parity error corrected
37 | doc=1993 | instruction cache parity error corrected
38 | doc=1994 | instruction cache parity e

In [31]:
candidates = candidate_pools["cache parity error"]

for rank, result in enumerate(candidates, start=1):
    if 46 <= rank <= 47:
        print(
            f"{rank} | "
            f"doc={result['doc_id']} | "
            f"{result['message']}"
        )

46 | doc=1505 | data cache search parity error detected. attempting to correct
47 | doc=1506 | data cache search parity error detected. attempting to correct


In [32]:
qrels["cache parity error"] = {

    # Strongly relevant:
    # Exact instruction cache parity error
    0: 2,
    1: 2,
    2: 2,
    3: 2,
    58: 2,
    59: 2,
    60: 2,
    61: 2,
    62: 2,
    63: 2,
    64: 2,
    65: 2,

    316: 2,
    317: 2,
    318: 2,
    319: 2,
    320: 2,
    321: 2,
    325: 2,
    326: 2,
    327: 2,
    328: 2,
    329: 2,
    330: 2,
    331: 2,
    333: 2,
    334: 2,
    346: 2,
    370: 2,
    372: 2,
    430: 2,

    1526: 2,
    1689: 2,
    1692: 2,
    1991: 2,
    1992: 2,
    1993: 2,
    1994: 2,
    1995: 2,
    1996: 2,
    1997: 2,
    1998: 2,

    # Somewhat relevant:
    # Data cache parity errors
    1502: 1,
    1503: 1,
    1504: 1,
    1505: 1,
    1506: 1,
    1507: 1,

    # Cache-related, but no parity error
    274: 1,
    275: 1,
    279: 1,

    # Irrelevant
    444: 0,
    445: 0,
    248: 0,
    272: 0,
}

In [33]:
judgments = qrels["cache parity error"]

print("Total:", len(judgments))
print(
    "Strong:",
    sum(v == 2 for v in judgments.values())
)
print(
    "Somewhat:",
    sum(v == 1 for v in judgments.values())
)
print(
    "Irrelevant:",
    sum(v == 0 for v in judgments.values())
)

Total: 55
Strong: 42
Somewhat: 9
Irrelevant: 4


In [34]:
candidates = candidate_pools["machine malfunction"]

print(
    f"machine malfunction: {len(candidates)} candidates"
)

machine malfunction: 55 candidates


In [35]:
for rank, result in enumerate(
    candidates,
    start=1
):
    if rank <= 25:
        print(
            f"{rank:2} | "
            f"doc={result['doc_id']} | "
            f"{result['message']}"
        )

 1 | doc=260 | machine check enable..............0
 2 | doc=261 | machine check enable..............0
 3 | doc=206 | machine check: i-fetch......................0
 4 | doc=207 | machine check: i-fetch......................0
 5 | doc=224 | machine check: i-fetch......................0
 6 | doc=239 | machine check: i-fetch......................0
 7 | doc=241 | machine check: i-fetch......................0
 8 | doc=1990 | Machine State Register: 0x0002f900
 9 | doc=283 | machine state register: 0x00002000
10 | doc=284 | machine state register: 0x00002000
11 | doc=287 | machine state register: 0x00002000
12 | doc=251 | machine state register: 0x00002000
13 | doc=1769 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
14 | doc=1770 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
15 | doc=1771 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
16 | doc=1772 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4

In [36]:
for rank, result in enumerate(
    candidates,
    start=1
):
    if 26 <= rank <= 55:
        print(
            f"{rank:2} | "
            f"doc={result['doc_id']} | "
            f"{result['message']}"
        )

26 | doc=449 | instruction address: 0x0062fe04
27 | doc=176 | instruction address: 0x00004ed8
28 | doc=200 | instruction address: 0x00004ed8
29 | doc=179 | instruction address: 0x00004ed8
30 | doc=175 | instruction address: 0x00004ed8
31 | doc=178 | instruction address: 0x00004ed8
32 | doc=230 | instruction address: 0x00004ed8
33 | doc=199 | instruction address: 0x00004ed8
34 | doc=197 | instruction address: 0x00004ed8
35 | doc=196 | instruction address: 0x00004ed8
36 | doc=177 | instruction address: 0x00004ed8
37 | doc=173 | instruction address: 0x00004ed8
38 | doc=216 | instruction address: 0x00004ed8
39 | doc=174 | instruction address: 0x00004ed8
40 | doc=214 | instruction address: 0x00004ed8
41 | doc=193 | instruction address: 0x00004ed8
42 | doc=198 | instruction address: 0x00004ed8
43 | doc=203 | instruction address: 0x00004ed8
44 | doc=195 | instruction address: 0x00004ed8
45 | doc=194 | instruction address: 0x00004ed8
46 | doc=202 | instruction address: 0x00004ed8
47 | doc=263 

In [37]:
qrels["machine malfunction"] = {

    # Strongly relevant
    1769: 2,
    1770: 2,
    1771: 2,
    1772: 2,
    1773: 2,
    1774: 2,
    1775: 2,

    450: 2,
    231: 2,
    232: 2,
    233: 2,
    234: 2,
    238: 2,

    263: 2,
    443: 2,

    # Somewhat relevant
    260: 1,
    261: 1,

    206: 1,
    207: 1,
    224: 1,
    239: 1,
    241: 1,

    1990: 1,
    283: 1,
    284: 1,
    287: 1,
    251: 1,

    449: 1,
    176: 1,
    200: 1,
    179: 1,
    175: 1,
    178: 1,
    230: 1,
    209: 1,
    197: 1,
    196: 1,
    177: 1,
    173: 1,
    216: 1,
    174: 1,
    214: 1,
    193: 1,
    198: 1,
    203: 1,
    195: 1,
    194: 1,
    202: 1,

    1975: 1,
    1979: 1,

    444: 1,
    445: 1,
    248: 1,
    272: 1,
}

In [38]:
judgments = qrels["machine malfunction"]

print("Total:", len(judgments))
print(
    "Strong:",
    sum(v == 2 for v in judgments.values())
)
print(
    "Somewhat:",
    sum(v == 1 for v in judgments.values())
)
print(
    "Irrelevant:",
    sum(v == 0 for v in judgments.values())
)

Total: 54
Strong: 15
Somewhat: 39
Irrelevant: 0


In [39]:
candidates = candidate_pools["machine malfunction"]

result = candidates[50]  # rank 51, because Python starts at 0

print(
    f"rank=51 | "
    f"doc={result['doc_id']} | "
    f"{result['message']}"
)

rank=51 | doc=1971 | 28 ddr error(s) detected and corrected on rank 0, symbol 21 over 11562 seconds


In [40]:
qrels["machine malfunction"][1971] = 1

In [41]:
judgments = qrels["machine malfunction"]

print("Total:", len(judgments))
print(
    "Strong:",
    sum(v == 2 for v in judgments.values())
)
print(
    "Somewhat:",
    sum(v == 1 for v in judgments.values())
)
print(
    "Irrelevant:",
    sum(v == 0 for v in judgments.values())
)

Total: 55
Strong: 15
Somewhat: 40
Irrelevant: 0


In [42]:
for query in qrels:

    print()
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    evaluation = evaluate_query(
        query,
        top_k=10
    )

    for system_name, metrics in evaluation.items():

        print(
            f"{system_name:10} | "
            f"P@10={metrics['precision']:.2f} | "
            f"nDCG@10={metrics['ndcg']:.4f}"
        )


QUERY: hardware stopped working
BM25       | P@10=0.00 | nDCG@10=0.0000
Semantic   | P@10=1.00 | nDCG@10=1.0000
RRF        | P@10=1.00 | nDCG@10=1.0000

QUERY: network connection failure
BM25       | P@10=1.00 | nDCG@10=1.0000
Semantic   | P@10=1.00 | nDCG@10=1.0000
RRF        | P@10=1.00 | nDCG@10=1.0000

QUERY: cache parity error
BM25       | P@10=1.00 | nDCG@10=1.0000
Semantic   | P@10=1.00 | nDCG@10=1.0000
RRF        | P@10=1.00 | nDCG@10=1.0000

QUERY: machine malfunction
BM25       | P@10=1.00 | nDCG@10=1.0000
Semantic   | P@10=1.00 | nDCG@10=0.7217
RRF        | P@10=1.00 | nDCG@10=0.7217


In [43]:
diagnostic_queries = [
    "hardware stopped working",
    "machine malfunction",
]

for query in diagnostic_queries:

    print()
    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    results = retrieve_all(
        query,
        top_k=10
    )

    for system_name in ["BM25", "Semantic", "RRF"]:

        print()
        print(f"--- {system_name} ---")

        for rank, result in enumerate(
            results[system_name],
            start=1
        ):
            relevance = qrels[query].get(
                result["doc_id"],
                0
            )

            print(
                f"{rank:2}. "
                f"doc={result['doc_id']} | "
                f"rel={relevance} | "
                f"{result['message'][:100]}"
            )


QUERY: hardware stopped working



--- BM25 ---

--- Semantic ---
 1. doc=1329 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset 
 2. doc=1219 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset 
 3. doc=1407 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset 
 4. doc=1216 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset 
 5. doc=1230 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset 
 6. doc=1227 | rel=2 | Node card is not fully functional
 7. doc=1229 | rel=2 | Node card is not fully functional
 8. doc=457 | rel=2 | Node card is not fully functional
 9. doc=620 | rel=2 | Node card is not fully functional
10. doc=1218 | rel=2 | Node card is not fully functional

--- RRF ---
 1. doc=1407 | rel=2 | Node card status: no ALERTs a

In [44]:
query = "hardware stopped working"

results = retrieve_all(
    query,
    top_k=10
)

print("=" * 80)
print("RRF DIAGNOSTIC")
print("=" * 80)

for rank, result in enumerate(results["RRF"], start=1):

    doc_id = result["doc_id"]
    relevance = qrels[query].get(doc_id, 0)

    print(
        f"{rank:2} | "
        f"doc={doc_id} | "
        f"rel={relevance} | "
        f"RRF={result['score']:.6f} | "
        f"{result['message'][:100]}"
    )

RRF DIAGNOSTIC
 1 | doc=1407 | rel=2 | RRF=0.016393 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset 
 2 | doc=1219 | rel=2 | RRF=0.016129 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset 
 3 | doc=1329 | rel=2 | RRF=0.015873 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset 
 4 | doc=1216 | rel=2 | RRF=0.015625 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset 
 5 | doc=1230 | rel=2 | RRF=0.015385 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset 
 6 | doc=1229 | rel=2 | RRF=0.015152 | Node card is not fully functional
 7 | doc=1227 | rel=2 | RRF=0.014925 | Node card is not fully functional
 8 | doc=620 | rel=2 | RRF=0.014706 | Node card is not fully functional
 9 | doc=1218 | rel=2 | RRF=0.014493 | Node card is not fully funct

In [45]:
query = "hardware stopped working"

results = retrieve_all(
    query,
    top_k=10
)

bm25_results = results["BM25"]
semantic_results = results["Semantic"]
rrf_results = results["RRF"]


bm25_ranks = {
    result["doc_id"]: rank
    for rank, result in enumerate(
        bm25_results,
        start=1
    )
}

semantic_ranks = {
    result["doc_id"]: rank
    for rank, result in enumerate(
        semantic_results,
        start=1
    )
}


print("=" * 80)
print("RRF COMPONENT DIAGNOSTIC")
print("=" * 80)

for rank, result in enumerate(
    rrf_results,
    start=1
):

    doc_id = result["doc_id"]

    print(
        f"{rank:2} | "
        f"doc={doc_id} | "
        f"BM25 rank={bm25_ranks.get(doc_id, '-'):>2} | "
        f"SEM rank={semantic_ranks.get(doc_id, '-'):>2} | "
        f"RRF={result['score']:.6f}"
    )

RRF COMPONENT DIAGNOSTIC
 1 | doc=1407 | BM25 rank= - | SEM rank= 3 | RRF=0.016393
 2 | doc=1219 | BM25 rank= - | SEM rank= 2 | RRF=0.016129
 3 | doc=1329 | BM25 rank= - | SEM rank= 1 | RRF=0.015873
 4 | doc=1216 | BM25 rank= - | SEM rank= 4 | RRF=0.015625
 5 | doc=1230 | BM25 rank= - | SEM rank= 5 | RRF=0.015385
 6 | doc=1229 | BM25 rank= - | SEM rank= 7 | RRF=0.015152
 7 | doc=1227 | BM25 rank= - | SEM rank= 6 | RRF=0.014925
 8 | doc=620 | BM25 rank= - | SEM rank= 9 | RRF=0.014706
 9 | doc=1218 | BM25 rank= - | SEM rank=10 | RRF=0.014493
10 | doc=1948 | BM25 rank= - | SEM rank= - | RRF=0.014286


In [46]:
for query in qrels:

    print()
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    evaluation = evaluate_query(
        query,
        top_k=10
    )

    for system_name, metrics in evaluation.items():

        print(
            f"{system_name:10} | "
            f"P@10={metrics['precision']:.2f} | "
            f"nDCG@10={metrics['ndcg']:.4f}"
        )


QUERY: hardware stopped working


BM25       | P@10=0.00 | nDCG@10=0.0000
Semantic   | P@10=1.00 | nDCG@10=1.0000
RRF        | P@10=1.00 | nDCG@10=1.0000

QUERY: network connection failure
BM25       | P@10=1.00 | nDCG@10=1.0000
Semantic   | P@10=1.00 | nDCG@10=1.0000
RRF        | P@10=1.00 | nDCG@10=1.0000

QUERY: cache parity error
BM25       | P@10=1.00 | nDCG@10=1.0000
Semantic   | P@10=1.00 | nDCG@10=1.0000
RRF        | P@10=1.00 | nDCG@10=1.0000

QUERY: machine malfunction
BM25       | P@10=1.00 | nDCG@10=1.0000
Semantic   | P@10=1.00 | nDCG@10=0.7217
RRF        | P@10=1.00 | nDCG@10=0.7217


In [47]:
for q, j in qrels.items():

    strong = sum(
        score == 2
        for score in j.values()
    )

    somewhat = sum(
        score == 1
        for score in j.values()
    )

    irrelevant = sum(
        score == 0
        for score in j.values()
    )

    print(
        f"{q:30} | "
        f"Strong : {strong:2} | "
        f"Somewhat : {somewhat:2} | "
        f"Irrelevant : {irrelevant:2} | "
        f"Total : {len(j):2}"
    )

hardware stopped working       | Strong : 18 | Somewhat : 30 | Irrelevant :  2 | Total : 50
network connection failure     | Strong : 12 | Somewhat :  0 | Irrelevant :  8 | Total : 20
cache parity error             | Strong : 42 | Somewhat :  9 | Irrelevant :  4 | Total : 55
machine malfunction            | Strong : 15 | Somewhat : 40 | Irrelevant :  0 | Total : 55


In [48]:
query = "machine malfunction"

bm25_results = engine.search_bm25(
    query,
    top_k=10
)

semantic_results = engine.search_semantic(
    query,
    top_k=10
)

hybrid_results = engine.search_hybrid(
    query,
    top_k=10
)

In [49]:
systems = {
    "BM25": bm25_results,
    "Semantic": semantic_results,
    "RRF": hybrid_results
}

print("=" * 70)
print(f"QUERY: {query}")
print("=" * 70)


for system_name, results in systems.items():

    print(f"\n--- {system_name} ---\n")

    for result in results[:10]:

        doc_id = result["doc_id"]

        relevance = qrels[query].get(
            doc_id,
            0
        )

        print(
            f"{result['rank']:2}. "
            f"{doc_id} | "
            f"rel={relevance} | "
            f"{result['message']}"
        )

QUERY: machine malfunction

--- BM25 ---

 1. 260 | rel=1 | machine check enable..............0
 2. 261 | rel=1 | machine check enable..............0
 3. 206 | rel=1 | machine check: i-fetch......................0
 4. 207 | rel=1 | machine check: i-fetch......................0
 5. 224 | rel=1 | machine check: i-fetch......................0
 6. 239 | rel=1 | machine check: i-fetch......................0
 7. 241 | rel=1 | machine check: i-fetch......................0
 8. 1990 | rel=1 | Machine State Register: 0x0002f900
 9. 283 | rel=1 | machine state register: 0x00002000
10. 284 | rel=1 | machine state register: 0x00002000

--- Semantic ---

 1. 251 | rel=1 | machine state register: 0x00002000
 2. 283 | rel=1 | machine state register: 0x00002000
 3. 287 | rel=1 | machine state register: 0x00002000
 4. 284 | rel=1 | machine state register: 0x00002000
 5. 261 | rel=1 | machine check enable..............0
 6. 260 | rel=1 | machine check enable..............0
 7. 1990 | rel=1 | Machine Stat

In [50]:
def get_ranked_doc_ids(results):
    """
    Extract document IDs from ranked search results.
    """

    return [
        result["doc_id"]
        for result in results
    ]

In [51]:
def precision_at_k(
        results,
        judgments,
        k=10
):
    """
    Calculate Precision@k

    A document is considered relevant
    if it's relevance is greater than 0.
    """


    top_results = results[:k]

    if not top_results:
        return 0.0

    relevant = sum(
        judgments.get(
            result["doc_id"],
            0
        ) > 0 
        for result in top_results
    )

    return relevant / len(top_results)

In [52]:
def recall_at_k(
        results,
        judgments,
        k=10
):

    """
    Calculate Recall@K

    Measures how many of all relevant judged documents
    were retreived in the top k.
    """

    top_results = results[:k]

    relevant_docs = {
        doc_id
        for doc_id, relevance
        in judgments.items()
        if relevance > 0
    }

    if not relevant_docs:
        return 0.0


    retrieved_relevant = sum(
        result["doc_id"] in relevant_docs
        for result in top_results
    )

    return (
        retrieved_relevant / len(relevant_docs)
    )

In [53]:
import math

def dcg_at_k(
    results,
    judgments,
    k=10
):
    """
    calculate Discounted Cumulative Gain.
    """

    dcg = 0.0

    for rank, result in enumerate(
        results[:k],
        start=1
    ):
        relevance = judgments.get(
            result["doc_id"], 0
        )

        gain = (
            (2 ** relevance) - 1
        )

        discount = math.log2(
            rank + 1
        )

        dcg += gain / discount

    return dcg

def ndcg_at_k(
        results,
        judgments,
        k = 10
):
    """
    Caculate Normalized Discounted
    Cumulative Gain at K.
    """

    actual_dcg = dcg_at_k(
        results,
        judgments,
        k
    )

    ideal_relevances = sorted(
        judgments.values(),
        reverse=True
    )[:k]

    ideal_dcg = 0.0

    for rank, relevance in enumerate(
        ideal_relevances,
        start=1
    ):

        gain = (
            (2 ** relevance) - 1
        )

        discount = math.log2(
            rank + 1
        )

        ideal_dcg += (
            gain / discount
        )

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg

In [54]:
query = "machine malfunction"

judgments = qrels[query]

bm25_results = engine.search_bm25(
    query, top_k=10
)

semantic_results = engine.search_semantic(
    query, top_k=10
)

hybrid_results = engine.search_hybrid(
    query, top_k=10
)

systems = {
    "BM25" : bm25_results,
    "Semantic" : semantic_results,
    "RRF" : hybrid_results
}

for name, res in systems.items():

    precision = precision_at_k(
        results,
        judgments,
        k=10
    )

    recall = recall_at_k(
        results,
        judgments,
        k=10
    )

    ndgc = ndcg_at_k(
        results,
        judgments,
        k=10
    )

    print(
        f"{name:10} | "
        f"P@10={precision:.4f} | "
        f"Recall@10={recall:.4f} | "
        f"nDGC@10={ndgc:.4f}"
    )

BM25       | P@10=1.0000 | Recall@10=0.1818 | nDGC@10=0.4662
Semantic   | P@10=1.0000 | Recall@10=0.1818 | nDGC@10=0.4662
RRF        | P@10=1.0000 | Recall@10=0.1818 | nDGC@10=0.4662


In [55]:
query = "machine malfunction"

judgments = qrels[query]

from collections import Counter

print(
    "Relevance distribution:",
    Counter(judgments.values())
)

Relevance distribution: Counter({1: 40, 2: 15})


In [56]:
print("\nTop 10 BM25 relevance scores:\n")

for result in bm25_results[:10]:

    doc_id = result["doc_id"]

    relevance = judgments.get(
        doc_id,
        0
    )

    print(
        f"rank={result['rank']:2} | "
        f"doc={doc_id} | "
        f"relevance={relevance}"
    )


Top 10 BM25 relevance scores:

rank= 1 | doc=260 | relevance=1
rank= 2 | doc=261 | relevance=1
rank= 3 | doc=206 | relevance=1
rank= 4 | doc=207 | relevance=1
rank= 5 | doc=224 | relevance=1
rank= 6 | doc=239 | relevance=1
rank= 7 | doc=241 | relevance=1
rank= 8 | doc=1990 | relevance=1
rank= 9 | doc=283 | relevance=1
rank=10 | doc=284 | relevance=1


In [57]:
queries = [
    "hardware stopped working",
    "network connection failure",
    "cache parity error",
    "machine malfunction"
]

systems = {
    "BM25": engine.search_bm25,
    "Semantic": engine.search_semantic,
    "RRF": engine.search_hybrid
}


for query in queries:

    print("\n" + "=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    judgments = qrels[query]

    for system_name, search_function in systems.items():

        results = search_function(
            query,
            top_k=10
        )

        precision = precision_at_k(
            results,
            judgments,
            k=10
        )

        recall = recall_at_k(
            results,
            judgments,
            k=10
        )

        ndcg = ndcg_at_k(
            results,
            judgments,
            k=10
        )

        print(
            f"{system_name:<10} | "
            f"P@10={precision:.4f} | "
            f"Recall@10={recall:.4f} | "
            f"nDCG@10={ndcg:.4f}"
        )


QUERY: hardware stopped working
BM25       | P@10=0.0000 | Recall@10=0.0000 | nDCG@10=0.0000
Semantic   | P@10=1.0000 | Recall@10=0.2083 | nDCG@10=1.0000
RRF        | P@10=1.0000 | Recall@10=0.2083 | nDCG@10=1.0000

QUERY: network connection failure
BM25       | P@10=1.0000 | Recall@10=0.8333 | nDCG@10=1.0000
Semantic   | P@10=1.0000 | Recall@10=0.8333 | nDCG@10=1.0000
RRF        | P@10=1.0000 | Recall@10=0.8333 | nDCG@10=1.0000

QUERY: cache parity error
BM25       | P@10=1.0000 | Recall@10=0.1961 | nDCG@10=1.0000
Semantic   | P@10=1.0000 | Recall@10=0.1961 | nDCG@10=1.0000
RRF        | P@10=1.0000 | Recall@10=0.1961 | nDCG@10=1.0000

QUERY: machine malfunction
BM25       | P@10=1.0000 | Recall@10=0.1818 | nDCG@10=0.3333
Semantic   | P@10=1.0000 | Recall@10=0.1818 | nDCG@10=0.4662
RRF        | P@10=1.0000 | Recall@10=0.1818 | nDCG@10=0.4662


In [58]:
from incidentiq.retrieval.query import (
    parse_query,
    get_scoring_terms
)

from incidentiq.retrieval.query import (
    parse_query,
    get_scoring_terms
)

In [59]:
query = "hardware stopped working"

parsed_query = parse_query(query)

terms = get_scoring_terms(
    parsed_query
)

print("PARSED QUERY:")
print(parsed_query)

print("\nSCORING TERMS:")
print(terms)

print("\nCANDIDATES PER TERM:")

for term in terms:

    candidates = engine.index.inverted_index.get(
        term,
        {}
    )

    print(
        f"{term:15} | "
        f"{len(candidates)} candidates"
    )

print("\nTOTAL BM25 CANDIDATES:")

all_candidates = engine.bm25.candidates(
    terms
)

print(len(all_candidates))

PARSED QUERY:
{'terms': ['hardware', 'stopped', 'working'], 'phrases': []}

SCORING TERMS:
['hardware', 'stopped', 'working']

CANDIDATES PER TERM:
hardware        | 0 candidates
stopped         | 0 candidates
working         | 0 candidates

TOTAL BM25 CANDIDATES:
0


In [60]:
print("Corpus size:")
print(len(engine.df))

print("\nFirst 5 messages:")
for message in engine.df["message"].head():
    print(message)

print("\nChecking corpus directly:")

for term in ["hardware", "stopped", "working"]:

    matches = engine.df[
        engine.df["message"]
        .str.lower()
        .str.contains(term, na=False)
    ]

    print(
        f"{term:10} | "
        f"{len(matches)} messages contain it"
    )

Corpus size:
2000

First 5 messages:
instruction cache parity error corrected
instruction cache parity error corrected
instruction cache parity error corrected
instruction cache parity error corrected
63543 double-hummer alignment exceptions

Checking corpus directly:
hardware   | 0 messages contain it
stopped    | 0 messages contain it
working    | 0 messages contain it


In [61]:
print("ENGINE DATAFRAME")
print(engine.df.shape)

print("\nCOLUMNS:")
print(engine.df.columns.tolist())

print("\nDATA SAMPLE:")
print(
    engine.df[
        [
            "log_id",
            "message"
        ]
    ].head(10)
)

ENGINE DATAFRAME
(2000, 8)

COLUMNS:
['log_id', 'label', 'timestamp', 'node', 'type', 'component', 'severity', 'message']

DATA SAMPLE:
   log_id                                            message
0       1           instruction cache parity error corrected
1       2           instruction cache parity error corrected
2       3           instruction cache parity error corrected
3       4           instruction cache parity error corrected
4       5           63543 double-hummer alignment exceptions
5       6             162 double-hummer alignment exceptions
6       7             141 double-hummer alignment exceptions
7       8                 CE sym 2, at 0x0b85eee0, mask 0x05
8       9  ciod: failed to read message prefix on control...
9      10  ciod: failed to read message prefix on control...


In [62]:
doc_id = 1239

print(
    engine.df.loc[
        doc_id
    ]
)

log_id                                                    1240
label                                                        -
timestamp                           2005-08-12 23:27:49.807991
node                                       R46-M0-N0-I:J18-U01
type                                                       RAS
component                                                  APP
severity                                                 FATAL
message      ciod: Error loading /bgl/apps/scaletest/perfor...
Name: 1239, dtype: object


In [63]:
query = "hardware stopped working"

judgments = qrels[query]

print(f"QUERY: {query}\n")

for doc_id, relevance in judgments.items():

    message = engine.df.loc[
        doc_id,
        "message"
    ]

    print(
        f"doc={doc_id} | "
        f"rel={relevance} | "
        f"{message}"
    )

QUERY: hardware stopped working

doc=1329 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is asserted. Temperature Mask is not active. No temperature error. Temperature Limit Error Latch is clear. PGOOD IS NOT ASSERTED. PGOOD ERROR LATCH IS ACTIVE. MPGOOD IS NOT OK. MPGOOD ERROR LATCH IS ACTIVE. The 2.5 volt rail is OK. The 1.5 volt rail is OK.
doc=1216 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is asserted. Temperature Mask is not active. No temperature error. Temperature Limit Error Latch is clear. PGOOD IS NOT ASSERTED. PGOOD ERROR LATCH IS ACTIVE. MPGOOD IS NOT OK. MPGOOD ERROR LATCH IS ACTIVE. The 2.5 volt rail is OK. The 1.5 volt rail is OK.
doc=1219 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is assert

In [64]:
query = "hardware stopped working"

judgments = qrels[query]

bm25_results = engine.search_bm25(
    query,
    top_k=10
)

semantic_results = engine.search_semantic(
    query,
    top_k=10
)

rrf_results = engine.search_hybrid(
    query,
    top_k=10
)


def inspect_overlap(
    name,
    results,
    judgments
):
    print(f"\n--- {name} ---\n")

    for result in results:

        doc_id = result["doc_id"]

        relevance = judgments.get(
            doc_id,
            0
        )

        print(
            f"doc={doc_id} | "
            f"rel={relevance} | "
            f"{result['message']}"
        )


inspect_overlap(
    "BM25",
    bm25_results,
    judgments
)

inspect_overlap(
    "Semantic",
    semantic_results,
    judgments
)

inspect_overlap(
    "RRF",
    rrf_results,
    judgments
)


--- BM25 ---


--- Semantic ---

doc=1329 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is asserted. Temperature Mask is not active. No temperature error. Temperature Limit Error Latch is clear. PGOOD IS NOT ASSERTED. PGOOD ERROR LATCH IS ACTIVE. MPGOOD IS NOT OK. MPGOOD ERROR LATCH IS ACTIVE. The 2.5 volt rail is OK. The 1.5 volt rail is OK.
doc=1219 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is asserted. Temperature Mask is not active. No temperature error. Temperature Limit Error Latch is clear. PGOOD IS NOT ASSERTED. PGOOD ERROR LATCH IS ACTIVE. MPGOOD IS NOT OK. MPGOOD ERROR LATCH IS ACTIVE. The 2.5 volt rail is OK. The 1.5 volt rail is OK.
doc=1407 | rel=2 | Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is asser

In [65]:
import pandas as pd


DATA_PATH = "../data/processed/logs.parquet"


df = pd.read_parquet(DATA_PATH)


print("Corpus size:", len(df))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 10 messages:\n")

for i, message in enumerate(
    df["message"].head(10),
    start=1
):
    print(f"{i}. {message}")

Corpus size: 2000

Columns:
['log_id', 'label', 'timestamp', 'node', 'type', 'component', 'severity', 'message']

First 10 messages:

1. instruction cache parity error corrected
2. instruction cache parity error corrected
3. instruction cache parity error corrected
4. instruction cache parity error corrected
5. 63543 double-hummer alignment exceptions
6. 162 double-hummer alignment exceptions
7. 141 double-hummer alignment exceptions
8. CE sym 2, at 0x0b85eee0, mask 0x05
9. ciod: failed to read message prefix on control stream (CioStream socket to 172.16.96.116:33569
10. ciod: failed to read message prefix on control stream (CioStream socket to 172.16.96.116:33370


In [66]:
message_counts = (
    df["message"]
    .value_counts()
)


print(
    "Unique messages:",
    len(message_counts)
)


print("\nMost common messages:\n")


for rank, (message, count) in enumerate(
    message_counts.head(30).items(),
    start=1
):
    print(
        f"{rank:2}. "
        f"[{count:3} occurrences] "
        f"{message}"
    )

Unique messages: 1367

Most common messages:

 1. [ 60 occurrences] data TLB error interrupt
 2. [ 51 occurrences] 0 microseconds spent in the rbs signal handler during 0 calls. 0 microseconds was the maximum time for a single instance of a correctable ddr.
 3. [ 42 occurrences] instruction cache parity error corrected
 4. [ 37 occurrences] 8 floating point alignment exceptions
 5. [ 35 occurrences] idoproxydb hit ASSERT condition: ASSERT expression=0 Source file=idotransportmgr.cpp Source line=1043 Function=int IdoTransportMgr::SendPacket(IdoUdpMgr*, BglCtlPavTrace*)
 6. [ 30 occurrences] data storage interrupt
 7. [ 27 occurrences] 1146800 double-hummer alignment exceptions
 8. [ 22 occurrences] 5 floating point alignment exceptions
 9. [ 20 occurrences] instruction address: 0x00004ed8
10. [ 15 occurrences] ciod: Error loading /bgl/apps/scaletest/performance/MINIBEN/mb_243_0810/allreduce.rts: invalid or missing program image, Exec format error
11. [ 15 occurrences] 2354412 floating p

In [67]:
search_terms = [
    "cache",
    "parity",
    "interrupt",
    "machine check",
    "node card",
    "floating point",
    "network",
    "timeout",
    "error detected",
    "exception"
]


for term in search_terms:

    matches = df[
        df["message"]
        .str.contains(
            term,
            case=False,
            na=False
        )
    ]

    print("\n" + "=" * 70)
    print(f"TERM: {term}")
    print(f"MATCHING LOGS: {len(matches)}")
    print("=" * 70)

    unique_messages = (
        matches["message"]
        .value_counts()
        .head(5)
    )

    for message, count in unique_messages.items():

        print(
            f"[{count}x] "
            f"{message}"
        )


TERM: cache
MATCHING LOGS: 54
[42x] instruction cache parity error corrected
[6x] data cache search parity error detected. attempting to correct
[2x] guaranteed instruction cache block touch.0
[2x] icache prefetch threshold................0
[1x] guaranteed data cache block touch........1

TERM: parity
MATCHING LOGS: 48
[42x] instruction cache parity error corrected
[6x] data cache search parity error detected. attempting to correct

TERM: interrupt
MATCHING LOGS: 209
[60x] data TLB error interrupt
[30x] data storage interrupt
[7x] 10722 total interrupts. 0 critical input interrupts. 0 microseconds total spent on critical input interrupts, 0 microseconds max time in a critical input interrupt.
[5x] program interrupt: illegal instruction......0
[5x] data store interrupt caused by dcbf.........0

TERM: machine check
MATCHING LOGS: 14
[7x] MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
[5x] machine check: i-fetch......................0
[2x] machine check enable.....

In [68]:
search_terms = [
    "machine check",
    "node card",
    "floating point",
    "network",
    "timeout",
    "error detected",
    "exception"
]


for term in search_terms:

    matches = df[
        df["message"]
        .str.contains(
            term,
            case=False,
            na=False
        )
    ]

    print("\n" + "=" * 70)
    print(f"TERM: {term}")
    print(f"MATCHING LOGS: {len(matches)}")
    print("-" * 70)

    unique_messages = (
        matches["message"]
        .value_counts()
        .head(10)
    )

    for rank, (message, count) in enumerate(
        unique_messages.items(),
        start=1
    ):

        print(
            f"{rank}. [{count}x] {message}"
        )


TERM: machine check
MATCHING LOGS: 14
----------------------------------------------------------------------
1. [7x] MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
2. [5x] machine check: i-fetch......................0
3. [2x] machine check enable..............0

TERM: node card
MATCHING LOGS: 28
----------------------------------------------------------------------
1. [6x] Node card is not fully functional
2. [6x] Can not get assembly information for node card
3. [5x] Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is asserted. Temperature Mask is not active. No temperature error. Temperature Limit Error Latch is clear. PGOOD IS NOT ASSERTED. PGOOD ERROR LATCH IS ACTIVE. MPGOOD IS NOT OK. MPGOOD ERROR LATCH IS ACTIVE. The 2.5 volt rail is OK. The 1.5 volt rail is OK.
4. [1x] Node card status: ALERT 0, ALERT 1, ALERT 2, ALERT 3 is (are) active. Clock Mode is Low. Clock Select is Midp

In [69]:
evaluation_queries = [
    "machine check timeout",
    "node card failure",
    "cache parity error",
    "network packet error",
    "floating point exception"
]

for query in evaluation_queries:
    print(query)

machine check timeout
node card failure
cache parity error
network packet error
floating point exception


In [70]:
def inspect_query_candidates(
    query,
    search_terms,
    max_messages=20
):
    """
    Find and display corpus messages matching
    one or more manually chosen search terms.
    """

    mask = False

    for term in search_terms:

        term_mask = (
            df["message"]
            .str.contains(
                term,
                case=False,
                na=False,
                regex=False
            )
        )

        mask = mask | term_mask

    matches = df[mask]

    print("=" * 80)
    print(f"QUERY: {query}")
    print(f"MATCHING DOCUMENTS: {len(matches)}")
    print("=" * 80)

    for doc_id, row in (
        matches
        .head(max_messages)
        .iterrows()
    ):

        print(
            f"\nDOC ID: {doc_id}"
        )

        print(
            row["message"]
        )

In [71]:
inspect_query_candidates(
    query="machine check timeout",
    search_terms=[
        "machine check",
        "timeout"
    ]
)

QUERY: machine check timeout
MATCHING DOCUMENTS: 14

DOC ID: 206
machine check: i-fetch......................0

DOC ID: 207
machine check: i-fetch......................0

DOC ID: 224
machine check: i-fetch......................0

DOC ID: 239
machine check: i-fetch......................0

DOC ID: 241
machine check: i-fetch......................0

DOC ID: 260
machine check enable..............0

DOC ID: 261
machine check enable..............0

DOC ID: 1769
MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)

DOC ID: 1770
MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)

DOC ID: 1771
MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)

DOC ID: 1772
MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)

DOC ID: 1773
MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)

DOC ID: 1774
MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)

DOC ID: 1775
MACHINE CHECK DCR read timeout (mc=e0

In [72]:
query = "machine check timeout"

judgments = {}

for doc_id, row in df.iterrows():

    message = row["message"].lower()

    # ----------------------------------------
    # Highly relevant
    # ----------------------------------------

    if (
        "machine check dcr read timeout"
        in message
    ):
        judgments[doc_id] = 2

    # ----------------------------------------
    # Somewhat relevant
    # ----------------------------------------

    elif "machine check" in message:

        judgments[doc_id] = 1


print("QUERY:", query)
print()

for doc_id, relevance in judgments.items():

    print(
        f"doc={doc_id} | "
        f"rel={relevance} | "
        f"{df.loc[doc_id, 'message']}"
    )

print()
print(
    "Total judgments:",
    len(judgments)
)

QUERY: machine check timeout

doc=206 | rel=1 | machine check: i-fetch......................0
doc=207 | rel=1 | machine check: i-fetch......................0
doc=224 | rel=1 | machine check: i-fetch......................0
doc=239 | rel=1 | machine check: i-fetch......................0
doc=241 | rel=1 | machine check: i-fetch......................0
doc=260 | rel=1 | machine check enable..............0
doc=261 | rel=1 | machine check enable..............0
doc=1769 | rel=2 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
doc=1770 | rel=2 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
doc=1771 | rel=2 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
doc=1772 | rel=2 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
doc=1773 | rel=2 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
doc=1774 | rel=2 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
doc=1775

In [73]:
inspect_query_candidates(
    query="node card failure",
    search_terms=[
        "node card"
    ],
    max_messages=30
)

QUERY: node card failure
MATCHING DOCUMENTS: 28

DOC ID: 457
Node card is not fully functional

DOC ID: 522
Can not get assembly information for node card

DOC ID: 620
Node card is not fully functional

DOC ID: 1202
Node card status: ALERT 0, ALERT 1, ALERT 2, ALERT 3 is (are) active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is not asserted. Temperature Mask is not active. No temperature error. Temperature Limit Error Latch is clear. PGOOD is asserted. PGOOD error latch is clear. MPGOOD is OK. MPGOOD error latch is clear. The 2.5 volt rail is OK. The 1.5 volt rail is OK.

DOC ID: 1203
Node card VPD check: U11 node in processor card slot J16 do not match. VPD ecid 04DF80A7942FFFFF0C081AE08CD2, found 0000000000000000000000000000

DOC ID: 1204
Can not get assembly information for node card

DOC ID: 1206
Can not get assembly information for node card

DOC ID: 1216
Node card status: no ALERTs are active. Clock Mode is Low. Clock Select is Midp

In [74]:
query = "node card failure"

judgments = {}

for doc_id, row in df.iterrows():

    message = row["message"].lower()

    if "node card is not fully functional" in message:
        judgments[doc_id] = 2

    elif (
        "can not get assembly information for node card"
        in message
    ):
        judgments[doc_id] = 2

    elif "node card vpd check" in message:
        judgments[doc_id] = 2

    elif "node card status" in message:
        judgments[doc_id] = 1


print("QUERY:", query)

for doc_id, relevance in judgments.items():
    print(
        f"doc={doc_id} | "
        f"rel={relevance} | "
        f"{df.loc[doc_id, 'message']}"
    )

print("\nRelevance distribution:")

from collections import Counter

print(Counter(judgments.values()))

print(
    "Total judgments:",
    len(judgments)
)

QUERY: node card failure
doc=457 | rel=2 | Node card is not fully functional
doc=522 | rel=2 | Can not get assembly information for node card
doc=620 | rel=2 | Node card is not fully functional
doc=1202 | rel=1 | Node card status: ALERT 0, ALERT 1, ALERT 2, ALERT 3 is (are) active. Clock Mode is Low. Clock Select is Midplane. Phy JTAG Reset is asserted. ASIC JTAG Reset is not asserted. Temperature Mask is not active. No temperature error. Temperature Limit Error Latch is clear. PGOOD is asserted. PGOOD error latch is clear. MPGOOD is OK. MPGOOD error latch is clear. The 2.5 volt rail is OK. The 1.5 volt rail is OK.
doc=1203 | rel=2 | Node card VPD check: U11 node in processor card slot J16 do not match. VPD ecid 04DF80A7942FFFFF0C081AE08CD2, found 0000000000000000000000000000
doc=1204 | rel=2 | Can not get assembly information for node card
doc=1206 | rel=2 | Can not get assembly information for node card
doc=1216 | rel=1 | Node card status: no ALERTs are active. Clock Mode is Low. Clo

In [75]:
inspect_query_candidates(
    query="cache parity error",
    search_terms=[
        "cache",
        "parity error"
    ],
    max_messages=30
)

QUERY: cache parity error
MATCHING DOCUMENTS: 54

DOC ID: 0
instruction cache parity error corrected

DOC ID: 1
instruction cache parity error corrected

DOC ID: 2
instruction cache parity error corrected

DOC ID: 3
instruction cache parity error corrected

DOC ID: 58
instruction cache parity error corrected

DOC ID: 59
instruction cache parity error corrected

DOC ID: 60
instruction cache parity error corrected

DOC ID: 61
instruction cache parity error corrected

DOC ID: 62
instruction cache parity error corrected

DOC ID: 63
instruction cache parity error corrected

DOC ID: 64
instruction cache parity error corrected

DOC ID: 65
instruction cache parity error corrected

DOC ID: 274
guaranteed instruction cache block touch.0

DOC ID: 275
guaranteed instruction cache block touch.0

DOC ID: 279
guaranteed data cache block touch........1

DOC ID: 282
icache prefetch depth....................0

DOC ID: 285
icache prefetch threshold................0

DOC ID: 286
icache prefetch threshold.

In [76]:
query = "cache parity error"

judgments = {}

for doc_id, row in df.iterrows():

    message = row["message"].lower()

    # ----------------------------------------
    # Highly relevant
    # ----------------------------------------

    if (
        "cache" in message
        and "parity error" in message
    ):
        judgments[doc_id] = 2

    # ----------------------------------------
    # Somewhat relevant
    # ----------------------------------------

    elif "cache" in message:
        judgments[doc_id] = 1


print("QUERY:", query)
print()

for doc_id, relevance in judgments.items():

    print(
        f"doc={doc_id} | "
        f"rel={relevance} | "
        f"{df.loc[doc_id, 'message']}"
    )

print("\nRelevance distribution:")

from collections import Counter

print(Counter(judgments.values()))

print(
    "Total judgments:",
    len(judgments)
)

QUERY: cache parity error

doc=0 | rel=2 | instruction cache parity error corrected
doc=1 | rel=2 | instruction cache parity error corrected
doc=2 | rel=2 | instruction cache parity error corrected
doc=3 | rel=2 | instruction cache parity error corrected
doc=58 | rel=2 | instruction cache parity error corrected
doc=59 | rel=2 | instruction cache parity error corrected
doc=60 | rel=2 | instruction cache parity error corrected
doc=61 | rel=2 | instruction cache parity error corrected
doc=62 | rel=2 | instruction cache parity error corrected
doc=63 | rel=2 | instruction cache parity error corrected
doc=64 | rel=2 | instruction cache parity error corrected
doc=65 | rel=2 | instruction cache parity error corrected
doc=274 | rel=1 | guaranteed instruction cache block touch.0
doc=275 | rel=1 | guaranteed instruction cache block touch.0
doc=279 | rel=1 | guaranteed data cache block touch........1
doc=282 | rel=1 | icache prefetch depth....................0
doc=285 | rel=1 | icache prefetch thr

In [77]:
inspect_query_candidates(
    query="network packet error",
    search_terms=[
        "network",
        "packet"
    ],
    max_messages=30
)

QUERY: network packet error
MATCHING DOCUMENTS: 41

DOC ID: 1207
idoproxydb hit ASSERT condition: ASSERT expression=0 Source file=idotransportmgr.cpp Source line=1043 Function=int IdoTransportMgr::SendPacket(IdoUdpMgr*, BglCtlPavTrace*)

DOC ID: 1208
idoproxydb hit ASSERT condition: ASSERT expression=0 Source file=idotransportmgr.cpp Source line=1043 Function=int IdoTransportMgr::SendPacket(IdoUdpMgr*, BglCtlPavTrace*)

DOC ID: 1209
idoproxydb hit ASSERT condition: ASSERT expression=0 Source file=idotransportmgr.cpp Source line=1043 Function=int IdoTransportMgr::SendPacket(IdoUdpMgr*, BglCtlPavTrace*)

DOC ID: 1210
idoproxydb hit ASSERT condition: ASSERT expression=0 Source file=idotransportmgr.cpp Source line=1043 Function=int IdoTransportMgr::SendPacket(IdoUdpMgr*, BglCtlPavTrace*)

DOC ID: 1211
idoproxydb hit ASSERT condition: ASSERT expression=0 Source file=idotransportmgr.cpp Source line=1043 Function=int IdoTransportMgr::SendPacket(IdoUdpMgr*, BglCtlPavTrace*)

DOC ID: 1212
idopr

In [78]:
query = "network packet error"

judgments = {}

for doc_id, row in df.iterrows():

    message = row["message"].lower()

    if (
        "error receiving packet on tree network"
        in message
    ):
        judgments[doc_id] = 2


print("QUERY:", query)
print()

for doc_id, relevance in judgments.items():

    print(
        f"doc={doc_id} | "
        f"rel={relevance} | "
        f"{df.loc[doc_id, 'message']}"
    )

print()
print("Relevance distribution:")

from collections import Counter

print(Counter(judgments.values()))

print(
    "Total judgments:",
    len(judgments)
)

QUERY: network packet error

doc=1520 | rel=2 | Error receiving packet on tree network, expecting type 57 instead of type 3 (softheader=0064588e 8aff0003 00000002 00000000) PSR0=00001f01 PSR1=00000000 PRXF=00000002 PIXF=00000007
doc=1718 | rel=2 | Error receiving packet on tree network, expecting type 57 instead of type 3 (softheader=00589370 90990003 00000002 00000000) PSR0=00001f01 PSR1=00000000 PRXF=00000002 PIXF=00000007
doc=1748 | rel=2 | Error receiving packet on tree network, expecting type 57 instead of type 3 (softheader=00ce22e8 e6200003 00000002 00000000) PSR0=00001f01 PSR1=00000000 PRXF=00000002 PIXF=00000007
doc=1749 | rel=2 | Error receiving packet on tree network, expecting type 57 instead of type 3 (softheader=00ce22e8 e6200003 00000002 00000000) PSR0=20021f01 PSR1=00000000 PRXF=00000002 PIXF=00000007
doc=1757 | rel=2 | Error receiving packet on tree network, expecting type 57 instead of type 3 (softheader=009756d5 8bfa0003 00000002 00000000) PSR0=20021f01 PSR1=00000000

In [79]:
inspect_query_candidates(
    query="floating point exception",
    search_terms=[
        "floating point",
        "exception"
    ],
    max_messages=30
)

QUERY: floating point exception
MATCHING DOCUMENTS: 256

DOC ID: 4
63543 double-hummer alignment exceptions

DOC ID: 5
162 double-hummer alignment exceptions

DOC ID: 6
141 double-hummer alignment exceptions

DOC ID: 231
exception syndrome register: 0x00800000

DOC ID: 232
exception syndrome register: 0x00800000

DOC ID: 233
exception syndrome register: 0x00800000

DOC ID: 234
exception syndrome register: 0x00800000

DOC ID: 238
exception syndrome register: 0x00800000

DOC ID: 248
program interrupt: imprecise exception......0

DOC ID: 256
floating point instr. enabled.....1

DOC ID: 257
floating point instr. enabled.....1

DOC ID: 258
floating point instr. enabled.....1

DOC ID: 266
byte ordering exception.....................0

DOC ID: 267
byte ordering exception.....................0

DOC ID: 270
program interrupt: imprecise exception......0

DOC ID: 272
program interrupt: imprecise exception......0

DOC ID: 292
floating point instr. enabled.....1

DOC ID: 373
199680 double-hummer al